#### Moving images from one directory to another

In [41]:
import os
import shutil
import pandas as pd
from tqdm import tqdm

# Load your df_b
df_b = pd.read_csv('/mnt/Internal/MedImage/chexpert_balanced_for_training_51_per_label_dis+demog+age.csv')
# Define original and new image roots
image_root_b = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"
image_root_a = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train_copy_generated_imgs/"

# Function to copy one file while preserving folder structure
def copy_preserve_structure(rel_path):
    # Clean up rel_path if it already starts with the folder name
    if rel_path.startswith("CheXpert-v1.0/train/"):
        rel_path = rel_path[len("CheXpert-v1.0/train/"):]

    src = os.path.join(image_root_b, rel_path)
    dst = os.path.join(image_root_a, rel_path)

    os.makedirs(os.path.dirname(dst), exist_ok=True)

    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        print(f"Warning: {src} does not exist.")

# Copy all df_b images into image_root_a
tqdm.pandas(desc="Copying images")
df_b['Path'].progress_apply(copy_preserve_structure)

# After copying, update df_b paths to now be rooted at image_root_a
# (or just store relative paths if you're planning to load from image_root_a)
df_b['Path'] = df_b['Path']  # still relative; no change needed if using image_root_a later

# Optionally save the updated df_b
df_b.to_csv("dataset_b_updated.csv", index=False)


Copying images: 100%|██████████| 1400/1400 [00:02<00:00, 478.67it/s]


#### Checking the size of a folder

In [42]:
directory = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train_copy_generated_imgs/"
import os
# Get the total size of all files and subfolders inside the folder
folder_size_mb = sum(os.path.getsize(os.path.join(root, f)) for root, dirs, files in os.walk(directory) for f in files) / (1024 * 1024)

print(f"Total size of '{directory}': {folder_size_mb:.2f} MB")


Total size of '/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train_copy_generated_imgs/': 12210.60 MB


#### Making CSV from Generated Dataset

In [18]:
import os
import pandas as pd

# Folder containing the images
base_folder = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train_copy_generated_imgs"

# List of 14 CheXpert disease labels
chexpert_labels = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

# Mapping of index ranges to disease labels
index_to_label = [
    (0, 500, "No Finding"),
    (501, 1000, "Enlarged Cardiomediastinum"),
    (1001, 1500, "Cardiomegaly"),
    (1501, 2000, "Lung Opacity"),
    (2001, 2500, "Lung Lesion"),
    (2501, 3000, "Edema"),
    (3001, 3500, "Consolidation"),
    (3501, 4000, "Pneumonia"),
    (4001, 4500, "Atelectasis"),
    (4501, 5000, "Pneumothorax"),
    (5001, 5500, "Pleural Effusion"),
    (5501, 6000, "Pleural Other"),
    (6001, 6500, "Fracture"),
    (6501, 7000, "Support Devices"),
    (7001, 8000, "No Finding"),
    (8001, 9000, "Enlarged Cardiomediastinum"),
    (9001, 10000, "Cardiomegaly"),
    (10001, 11000, "Lung Opacity"),
    (11001, 12000, "Lung Lesion"),
    (12001, 13000, "Edema"),
    (13001, 14000, "Consolidation"),
    (14001, 15000, "Pneumonia"),
    (15001, 16000, "Atelectasis"),
    (16001, 17000, "Pneumothorax"),
    (17001, 18000, "Pleural Effusion"),
    (18001, 19000, "Pleural Other"),
    (19001, 20000, "Fracture"),
    (21001, 22000, "Support Devices"),
]

# Function to get label from index
def get_label_from_index(index):
    for start, end, label in index_to_label:
        if start <= index <= end:
            return label
    return "Unknown"

# Walk through the directory and collect all images
image_paths = []
for root, _, files in os.walk(base_folder):
    for file in files:
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, base_folder)
            chexpert_style_path = os.path.join("CheXpert-v1.0/train", rel_path).replace("\\", "/")
            image_paths.append(chexpert_style_path)

# Sort the paths so indexing is consistent
image_paths.sort()

# Prepare rows with one-hot encoded disease labels
data_rows = []
for idx, path in enumerate(image_paths):
    label = get_label_from_index(idx)
    row = {"Path": path}
    for disease in chexpert_labels:
        row[disease] = 1 if disease == label else 0
    data_rows.append(row)

# Create the DataFrame and save to CSV
df = pd.DataFrame(data_rows)
df.to_csv("generated_chexpert_style_multilabel.csv", index=False)
print(f"CSV saved with {len(df)} entries and one-hot disease columns ✅")


CSV saved with 24968 entries and one-hot disease columns ✅


### Combining CSV's

In [43]:
import pandas as pd

# Load both CSVs
# df_A = pd.read_csv("/home/dawood/lab2_rotaion/generated_chexpert_style_multilabel.csv")
# df_B = pd.read_csv("/mnt/Internal/MedImage/chexpert_balanced_300_per_label.csv")
df_A = pd.read_csv("/home/dawood/lab2_rotaion/combined_dataset.csv")
df_B = pd.read_csv('/mnt/Internal/MedImage/chexpert_balanced_for_training_51_per_label_dis+demog+age.csv')

# Set 'Path' as index so we can do an efficient update
df_A.set_index("Path", inplace=True)
df_B.set_index("Path", inplace=True)

# Update A with B (overwrite matching rows)
df_A.update(df_B)

# Concatenate A and the new B rows that are not in A
df_combined = pd.concat([df_A, df_B[~df_B.index.isin(df_A.index)]])

# Reset index to get 'Path' back as a column
df_combined.reset_index(inplace=True)

# Save the final combined CSV
df_combined.to_csv("combined_dataset.csv", index=False)
print(f"✅ Final CSV saved with {len(df_combined)} unique images to 'combined_dataset.csv'")


/tmp/ipykernel_483361/2944400017.py:6: DtypeWarning: Columns (20,22,23,26,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df_A = pd.read_csv("/home/dawood/lab2_rotaion/combined_dataset.csv")


✅ Final CSV saved with 30264 unique images to 'combined_dataset.csv'


In [37]:
df = pd.read_csv("/home/dawood/lab2_rotaion/generated_chexpert_style_multilabel.csv")
df.head(1000)

,Path,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0/train/patient00001/study1/view1_...,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,CheXpert-v1.0/train/patient00002/study1/view1_...,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,CheXpert-v1.0/train/patient00002/study2/view1_...,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,CheXpert-v1.0/train/patient00003/study1/view1_...,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,CheXpert-v1.0/train/patient00004/study1/view1_...,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CheXpert-v1.0/train/patient00307/study1/view1_...,0,1,0,0,0,0,0,0,0,0,0,0,0,0
996,CheXpert-v1.0/train/patient00307/study2/view1_...,0,1,0,0,0,0,0,0,0,0,0,0,0,0
997,CheXpert-v1.0/train/patient00307/study3/view1_...,0,1,0,0,0,0,0,0,0,0,0,0,0,0
998,CheXpert-v1.0/train/patient00308/study2/view1_...,0,1,0,0,0,0,0,0,0,0,0,0,0,0
